**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Capstone: Build a Full System

Not new theory — proof that the curriculum *composes*. One pipeline, end to end: IQ capture → detection (matched filter + CFAR) → tracking (Kalman) → classification (CNN) → results database. Every stage is a workshop you've taken; here they hold hands. Runs on synthetic IQ so it executes anywhere; swap in an [RTL-SDR capture](../Intro_SDR/Software_Defined_Radio.ipynb) and nothing else changes.

## 1. Pre-requisites

The whole curriculum, honestly — minimally: [Statistical SP](../Intro_DSP/Statistical_Signal_Processing.ipynb), [Kalman](../Intro_Time_Series/Intro_AdFilt_KF.ipynb), [CNN](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb), [Databases](../Intro_Host_Prog/Intro_Databases/Intro_Databases.ipynb).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import sqlite3
import matplotlib.pyplot as plt
from scipy import signal as sig
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *The Scenario & the Signal Generator* (~30 min)
**Goal:** an emitter moves through a noisy band, transmitting bursts; we build the world to be monitored.
**Feeds into:** Session 2 (detection).

---

<details><summary>🎓 <b>Teacher notes — Session 1: The Scenario & the Signal Generator</b></summary>

**Open by naming what this notebook is proving, since it's easy to mistake for "one more DSP demo":** nothing here is new theory — every stage (detection, tracking, classification, storage) is a workshop students already took. The entire point is showing those pieces *compose* into a system, and that the interfaces between stages are exactly the kind of place a student's own project could swap in something smarter.

**"Because we built the world, every stage gets an oracle" is the single most important methodological point in the whole capstone — return to it repeatedly.** Every later session's evaluation (hit rate, RMSE, classifier accuracy, the final verdict) is only checkable because this session planted ground truth: `f_true` is the actual carrier trajectory, `MOD` is the actual modulation, `burst_starts` are the actual burst times. In a real deployment none of this exists — you'd never know your detector's hit rate or your tracker's RMSE with this precision. Make sure students register that synthetic data isn't just "easier," it's the *only* way this capstone can grade itself.

**The signal model itself deserves a slow walk, since three separate ideas are stacked in one cell:** a random-walk carrier (`f_true`, cumulative Gaussian steps — the same random-walk structure as a Wiener process from probability, now playing the role of frequency drift) models a real emitter's imperfect oscillator; short bursts with silent gaps between them models a real duty-cycled or scanning transmitter rather than a continuous carrier; and additive complex Gaussian noise (`iq`, scaled by $\sqrt{0.5}$ per real/imaginary component so total noise power is 1) sets the noise floor everything else must fight. Ask students what would change if `MOD` were randomized per-burst instead of fixed per-emitter — that's exactly the harder scenario Session 4's classifier has to solve in general, even though this specific run uses one fixed type.
</details>

## 2. The World

💡 **Intuition.** System design starts with the truth you'll grade yourself against. Our world: an emitter drifts across the band (a slowly-moving carrier frequency), transmitting short bursts of one of three modulation types, buried in noise. The pipeline must find the bursts, track the drift, and identify the modulation — and because we built the world, every stage gets an oracle.

In [2]:
fs = 100_000
DUR, BURST, GAP = 4.0, 0.02, 0.05
n_samp = int(DUR*fs)
t_all = np.arange(n_samp)/fs

# emitter truth: carrier random-walks; burst type fixed per emitter
f_true = 20_000 + np.cumsum(rng.normal(0, 30, n_samp))          # drifting carrier [Hz]
MOD = "chirp"                                                    # this emitter's fingerprint
burst_starts = np.arange(0.1, DUR-0.1, BURST+GAP)

iq = (rng.standard_normal(n_samp) + 1j*rng.standard_normal(n_samp)) * np.sqrt(0.5)  # noise floor
n_burst = int(BURST*fs)
for bs in burst_starts:
    i0 = int(bs*fs)
    tt = np.arange(n_burst)/fs
    fc = f_true[i0]
    if MOD == "chirp":  base = sig.chirp(tt, 0, BURST, 3000)
    env = np.exp(2j*np.pi*fc*tt) * base
    iq[i0:i0+n_burst] += 4.0 * env
print(f"{len(burst_starts)} bursts planted; carrier wanders {f_true.min()/1e3:.1f} → {f_true.max()/1e3:.1f} kHz")

55 bursts planted; carrier wanders 13.5 → 35.1 kHz


**What just happened.** `burst_starts` planted 55 bursts across the 4-second window (one every `BURST+GAP` = 0.07 s, minus edge margins), and the printed carrier range (13.5–35.1 kHz) is the realized excursion of a random walk that started at 20 kHz — Gaussian steps of $\sigma=30$ Hz accumulate over 400,000 samples, so the carrier is free to drift several kHz in either direction over the full recording. This random-walk carrier is deliberately the hard part of the scenario: every downstream stage has to cope with a target that doesn't sit still in frequency, which is exactly why Session 3 needs a Kalman filter rather than a fixed frequency estimate.

---
### 🕐 Session 2 of 4 — *Detection: Energy → CFAR* (~35 min)
**Goal:** find the bursts in time-frequency; CFAR keeps false alarms honest.
**Builds on:** [Statistical SP](../Intro_DSP/Statistical_Signal_Processing.ipynb) S4; [Radar](../Intro_DSP/Radar_Signal_Processing.ipynb) S3. &nbsp; **Feeds into:** Session 3 (tracking).

---

<details><summary>🎓 <b>Teacher notes — Session 2: Detection: Energy → CFAR</b></summary>

**This session is a direct reuse of two prior workshops fused together — call out both explicitly.** The STFT-then-collapse-to-a-per-frame-statistic step is Statistical Signal Processing's energy-detection material; the CFAR loop is lifted "verbatim" (per the code comment) from the Radar workshop. Nothing here is new — the achievement is wiring an existing detector onto this specific scenario's data shape.

**`return_onesided=False` in the STFT call is a detail worth flagging even though it passes silently** — this is complex IQ data, not a real-valued signal, so the negative-frequency half carries independent information (it's where the frequency-mirroring trick used later, `f_ if f_ < fs/2 else f_ - fs`, comes from) rather than being a redundant mirror of the positive half. Getting this flag wrong for IQ data silently throws away half the spectrum.

**Walk the CFAR loop as "adaptive thresholding," not a fixed cutoff — this is the concept the Radar workshop introduced and this session assumes.** The threshold at each time index is `scale * median` of a *local* reference window (excluding a guard band around the cell under test, to avoid the burst itself polluting its own threshold) — so the detector adapts to locally varying noise power instead of using one global cutoff that would either miss quiet stretches or false-alarm on loud ones. Ask why the guard cells exist at all: without them, the reference window would include the very samples ramping into/out of the burst under test, inflating the threshold right where you need it lowest.

**The event-grouping logic (collapsing consecutive `True` detections into one event, keeping the peak) is worth a beat of its own:** a burst spans many STFT frames, so naive per-frame detection would report the same burst multiple times — this is the deduplication step that turns "which frames exceeded threshold" into "how many distinct events occurred," which is the actual quantity Session 3's tracker needs one measurement per burst, not several.
</details>

In [3]:
f_stft, t_stft, Z = sig.stft(iq, fs=fs, nperseg=1024, noverlap=768, return_onesided=False)
P = np.abs(Z)**2
# collapse to a detection statistic per frame: max power over frequency
stat = P.max(0)
# CA-CFAR along time (from the Radar workshop, verbatim idea)
n_ref, n_guard, scale = 24, 4, 7.0
th = np.full_like(stat, np.inf)
for i in range(n_ref+n_guard, len(stat)-n_ref-n_guard):
    ref = np.r_[stat[i-n_ref-n_guard:i-n_guard], stat[i+n_guard+1:i+n_guard+1+n_ref]]
    th[i] = scale*np.median(ref)                       # median-CFAR: robust to burst pollution
det = stat > th
# group consecutive detections into events; record their peak frequency
events = []
i = 0
while i < len(det):
    if det[i]:
        j = i
        while j < len(det) and det[j]: j += 1
        k = i + np.argmax(stat[i:j])
        f_peak = f_stft[np.argmax(P[:, k])]
        events.append((t_stft[k], f_peak % fs))
        i = j
    else: i += 1
events = [(tt_, f_ if f_ < fs/2 else f_ - fs) for tt_, f_ in events]

hits = sum(any(abs(tt_ - (bs+BURST/2)) < 0.03 for tt_, _ in events) for bs in burst_starts)
print(f"planted bursts: {len(burst_starts)}   detected events: {len(events)}   planted-burst hit rate: {hits/len(burst_starts):.0%}")

planted bursts: 55   detected events: 55   planted-burst hit rate: 100%


**What just happened.** CFAR found all 55 planted bursts with zero missed detections and zero spurious events (55 detected events, 100% hit rate) — a clean result that reflects a genuinely easy detection problem: each burst is injected at amplitude 4.0 against a unit-variance noise floor, a very high SNR by design, so this scenario is testing the *pipeline plumbing*, not detector sensitivity at the edge of noise. The median-based CA-CFAR threshold (`scale*median(ref)` rather than the more common mean-based estimator) is the deliberate choice here: a mean-based reference window can itself get pulled upward by a burst sitting inside the reference cells, inflating the threshold and causing a miss right next to a strong signal — the median is far more robust to exactly that kind of self-pollution, which matters once bursts start arriving close together in time.

---
### 🕐 Session 3 of 4 — *Tracking the Drift: Kalman* (~35 min)
**Goal:** the detections are noisy frequency snapshots; a Kalman filter strings them into a track.
**Builds on:** [Kalman workshop](../Intro_Time_Series/Intro_AdFilt_KF.ipynb). &nbsp; **Feeds into:** Session 4 (classification & the database).

---

<details><summary>🎓 <b>Teacher notes — Session 3: Tracking the Drift: Kalman</b></summary>

**Frame this session as "turning a pile of independent snapshots into one coherent story" — that's the entire job of a tracker.** Session 2 produced 55 isolated (time, frequency) detections with no memory of each other; this session's Kalman filter is the first thing that treats them as *one emitter's trajectory* rather than 55 unrelated events. If a student asks "why not just use the raw detections," the honest answer is in the plot: raw frequency estimates are noisier than the smoothed track, and the tracker also gives you a *velocity* estimate (drift rate) that no single detection can provide on its own.

**The state vector `[frequency, drift-rate]` and constant-velocity model `F` are worth deriving from first principles rather than presenting as boilerplate:** `F = [[1, dt], [0, 1]]` says "predicted frequency = old frequency + drift_rate × dt; predicted drift_rate = unchanged" — the standard constant-velocity kinematic model, here applied to *frequency* as the tracked quantity instead of position. This is the same Kalman machinery from the Kalman workshop, with the physical interpretation swapped from "position/velocity in space" to "carrier frequency/drift rate in the spectrum."

**`Q` and `R` are the two numbers that actually control the filter's behavior — spend real time on what they mean, since tuning them is the practical skill.** `R` (measurement noise) encodes "how much do I trust a single CFAR frequency estimate" — larger `R` means the filter leans more on its own prediction and less on each new measurement. `Q` (process noise) encodes "how much can the true state itself change between measurements" — the large drift-rate entry in `Q` here specifically tells the filter to expect the carrier's velocity to wander (matching the random-walk generator from Session 1), which is why the filter can track a genuinely non-constant-velocity signal despite using a constant-velocity model: `Q` gives it permission to keep adapting.

**The downmixing payoff arrives in Session 4 — plant that connection now.** This session's track isn't just a nicer plot: Session 4 uses `track`'s frequency estimate to mix each burst down to baseband before classification. A track with poor RMSE would hand the classifier misaligned spectrograms — tracking quality directly gates classification quality, which is exactly the kind of cross-stage dependency this capstone exists to make visible.
</details>

In [4]:
# constant-velocity Kalman on (frequency, drift-rate), fed by the detected events
meas = sorted(events)
dt_meas = np.diff([m[0] for m in meas]).mean()
F = np.array([[1, dt_meas], [0, 1]])
Hm = np.array([[1.0, 0.0]])
Q = np.diag([10.0, 3000.0]); R = np.array([[80_000.0]])
x_kf = np.array([meas[0][1], 0.0]); Pk = np.diag([1e6, 1e6])
track = []
for tt_, f_meas in meas:
    x_kf = F @ x_kf; Pk = F @ Pk @ F.T + Q
    S = Hm @ Pk @ Hm.T + R
    K = Pk @ Hm.T / S
    x_kf = x_kf + (K * (f_meas - Hm @ x_kf)).ravel()
    Pk = (np.eye(2) - K @ Hm) @ Pk
    track.append((tt_, x_kf[0]))

track_t = np.array([a for a, _ in track]); track_f = np.array([b for _, b in track])
truth_at = np.interp(track_t, t_all, f_true)
rmse_raw = np.sqrt(np.mean((np.array([f_ for _, f_ in meas]) - truth_at)**2))
rmse_kf = np.sqrt(np.mean((track_f - truth_at)**2))
plt.figure(figsize=(9, 2.8))
plt.plot(t_all[::100], f_true[::100]/1e3, "k--", linewidth=1, label="true carrier")
plt.plot([m[0] for m in meas], [m[1]/1e3 for m in meas], ".", alpha=0.5, label="detections")
plt.plot(track_t, track_f/1e3, "C1", label="Kalman track")
plt.legend(fontsize=8); plt.xlabel("time [s]"); plt.ylabel("kHz")
plt.title(f"raw detections RMSE {rmse_raw:.0f} Hz → Kalman track RMSE {rmse_kf:.0f} Hz")
plt.tight_layout(); plt.show()

/tmp/ipykernel_321604/2008688132.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The plot title reports the raw per-detection frequency error against the Kalman-smoothed track's error — the Kalman filter should show a clear RMSE reduction, and here's why: each individual CFAR detection's peak-frequency estimate is quantized by the STFT's frequency-bin resolution (`fs/nperseg` ≈ 98 Hz per bin at `nperseg=1024`) and corrupted by that frame's noise realization, while the Kalman filter combines the constant-velocity motion model (`F`) with every measurement seen so far, averaging down noise that's independent frame-to-frame while still tracking genuine drift via the velocity state. The process noise `Q` and measurement noise `R` set how much the filter trusts the model vs. the data — `Q`'s large drift-rate entry (3000) tells the filter the carrier's velocity can itself change quickly (matching the random-walk carrier), while `R` (80,000 Hz²) encodes how noisy a single CFAR frequency estimate is expected to be.

---
### 🕐 Session 4 of 4 — *Classification & the Ledger* (~40 min)
**Goal:** a CNN identifies each burst's modulation; every verdict lands in a queryable database.
**Builds on:** [CNN](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb); [Databases](../Intro_Host_Prog/Intro_Databases/Intro_Databases.ipynb).

---

<details><summary>🎓 <b>Teacher notes — Session 4: Classification & the Ledger</b></summary>

**This session closes the loop, but it's also where the capstone's most important lesson hides — flag it before running the cell, not after.** Students will see "classifier holdout accuracy: 100%" and be tempted to declare victory. Hold off: the debrief on the results cell walks through why that 100% number and the pipeline's actual on-deployment performance (~58% of votes correctly say "chirp," even though every planted burst *is* chirp) are two different things measuring two different distributions. This is arguably the single most transferable lesson in the whole capstone — a model's reported accuracy is only as trustworthy as how well its test set matches deployment conditions.

**Walk `burst_example`'s three synthetic modulations as intentionally distinguishable, since that's what makes 100% holdout accuracy unsurprising on its own terms:** a chirp (linear frequency sweep), a BPSK-like phase-flip waveform, and an FM tone are spectrally very different shapes in a log-magnitude spectrogram — this is a comfortably separable 3-class problem for a small CNN, which is *why* holdout accuracy alone doesn't tell you much about robustness. The interesting question isn't "can the CNN tell these apart," it's "does the CNN see the same *kind* of input at inference time that it saw during training."

**The residual-offset detail (`df`, added to simulate tracking error) is the crux of the train/deployment mismatch, and it's worth making concrete with a number.** Training bursts get a random frequency offset drawn once per example; deployed bursts get downmixed using whatever frequency error Session 3's Kalman track actually produced for that specific burst. If those two error distributions don't match — different variance, different structure (Kalman error is correlated in time, the training offset is i.i.d.) — the classifier is being asked to generalize outside what it trained on, invisibly.

**Close on the database, since it's easy to treat as a footnote after the more exciting ML content:** `db.execute(... GROUP BY verdict ...)` is the Databases workshop's aggregation material paying off exactly as advertised — every individual verdict and its confidence is preserved in queryable form, which is precisely what let the debrief above catch the confidence-column anomaly (bpsk's mean confidence *higher* than chirp's, despite being wrong every time). A system that only logged the final majority-vote string would have hidden that signal entirely.
</details>

In [5]:
# train the classifier on synthetic bursts of the 3 candidate types (baseband spectrograms)
def burst_example(kind):
    tt = np.arange(n_burst)/fs
    df = rng.normal(0, 150)                                  # residual tracking error, like the pipeline's
    if kind == 0: base = sig.chirp(tt, 0, BURST, 3000)
    elif kind == 1: base = np.sign(np.sin(2*np.pi*400*tt))*np.cos(2*np.pi*1500*tt)   # BPSK-ish
    else: base = np.cos(2*np.pi*(1500 + 800*np.sin(2*np.pi*60*tt))*tt)               # FM tone
    x = 4.0*base*np.cos(2*np.pi*df*tt) + 1.0*rng.standard_normal(n_burst)   # match pipeline amplitude, noise, AND residual offset
    _, _, S = sig.stft(x, fs=fs, nperseg=128)
    S = np.log1p(np.abs(S))[:32, :14]
    return (S - S.mean())/(S.std()+1e-6)

Xc = torch.tensor(np.stack([burst_example(k % 3) for k in range(1200)]), dtype=torch.float32)[:, None]
yc = torch.tensor([k % 3 for k in range(1200)])
cnn = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Flatten(), nn.Linear(8*16*7, 3))
opt = torch.optim.Adam(cnn.parameters(), lr=2e-3)
for ep in range(6):
    for i in range(0, 1000, 64):
        opt.zero_grad(); nn.functional.cross_entropy(cnn(Xc[i:i+64]), yc[i:i+64]).backward(); opt.step()
acc = (cnn(Xc[1000:]).argmax(1) == yc[1000:]).float().mean()
print(f"classifier holdout accuracy: {acc:.0%}")

# run it on the DETECTED bursts (mix each event down to baseband using the KALMAN track)
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE detections (t REAL, f_hz REAL, verdict TEXT, confidence REAL)")
names = ["chirp", "bpsk", "fm"]
votes = []
with torch.no_grad():
    for tt_, f_tr in track:
        i0 = max(0, int(tt_*fs) - n_burst//2)
        seg = iq[i0:i0+n_burst]
        if len(seg) < n_burst: continue
        bb = np.real(seg * np.exp(-2j*np.pi*f_tr*np.arange(len(seg))/fs))   # Kalman-guided downmix
        _, _, S = sig.stft(bb, fs=fs, nperseg=128)
        S = np.log1p(np.abs(S))[:32, :14]; S = (S - S.mean())/(S.std()+1e-6)
        logits = cnn(torch.tensor(S, dtype=torch.float32)[None, None])
        p = torch.softmax(logits, 1)[0]
        votes.append(int(p.argmax()))
        db.execute("INSERT INTO detections VALUES (?,?,?,?)", (tt_, f_tr, names[p.argmax()], float(p.max())))
db.commit()

print("\nthe ledger answers questions (the Databases workshop, cashing in):")
for row in db.execute("SELECT verdict, COUNT(*), AVG(confidence) FROM detections GROUP BY verdict ORDER BY 2 DESC"):
    print(f"  {row[0]:6s}: {row[1]:3d} detections, mean confidence {row[2]:.2f}")
maj = names[max(set(votes), key=votes.count)]
print(f"\nsystem verdict on the emitter: '{maj}'   (planted truth: 'chirp')")

classifier holdout accuracy: 100%

the ledger answers questions (the Databases workshop, cashing in):
  chirp :  32 detections, mean confidence 0.69
  bpsk  :  19 detections, mean confidence 0.78
  fm    :   4 detections, mean confidence 0.50

system verdict on the emitter: 'chirp'   (planted truth: 'chirp')


**What just happened — and it's worth reading past the headline number.** The CNN hit 100% holdout accuracy on synthetic training-distribution bursts, but every one of the 55 real bursts in this scenario is planted as `chirp` — and the pipeline's own votes split 32 chirp / 19 bpsk / 4 fm, only ~58% "chirp." That gap between 100% holdout accuracy and ~58% correct-on-deployment is not a contradiction, it's a real and common failure mode: the *training* distribution (`burst_example`) adds a random residual frequency offset (`df`, $\sigma=150$ Hz) to simulate tracking error, but the *actual* bursts fed to the classifier are downmixed using the Kalman-tracked frequency, whose true residual error depends on the track quality computed two cells up — if that residual error, timing alignment, or noise realization differs from what `burst_example` simulated, the classifier is evaluating on a distribution it wasn't quite trained for. The majority vote still recovers the correct system-level verdict ('chirp', matching planted truth) because 32 is a plurality, but the *confidence* column tells the same story: chirp's mean confidence (0.69) is actually *lower* than the wrong-every-time bpsk verdict's (0.78) — a reminder that a per-detection classifier being right on average doesn't mean any single verdict, or its confidence score, should be trusted at face value. That's exactly why the ledger stores every vote instead of collapsing straight to one number.

## 3. Conclusion

Detection found the bursts (hit rate printed), the Kalman track cut the frequency error and *steered the downmixer*, the CNN identified the modulation, and SQL holds the evidence. No stage is new; the composition is the achievement — and every interface between stages is a place your own projects can swap in smarter pieces.

**The real-hardware version:** replace Session 1 with an [RTL-SDR capture](../Intro_SDR/Software_Defined_Radio.ipynb); budget the pipeline with [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb); containerize it with [Containers](../Intro_Host_Prog/Intro_Containers/Intro_Containers.ipynb). That's a senior-project-grade system, from parts you already own.

---
**This is the final playlist.** Where next is yours.